In [ ]:
!pip install imbalanced-learn xgboost joblib -q

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving UCI_Credit_Card.csv to UCI_Credit_Card.csv


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

from imblearn.over_sampling import SMOTE

# ==========================================================
# LOAD DATASET
# ==========================================================

df = pd.read_csv("UCI_Credit_Card.csv")

print("="*60)
print("DATASET INFORMATION")
print("="*60)

print("Shape:", df.shape)

print("\nFirst 5 Rows")
print(df.head())


df.drop("ID", axis=1, inplace=True)

# ==========================================================
# MISSING VALUES
# ==========================================================

print("\nMissing Values")
print(df.isnull().sum())

# ==========================================================
# TARGET DISTRIBUTION
# ==========================================================

print("\nTarget Distribution")
print(df["default.payment.next.month"].value_counts())

plt.figure(figsize=(6,4))
sns.countplot(
    x="default.payment.next.month",
    data=df
)
plt.title("Class Distribution Before SMOTE")
plt.show()

# ==========================================================
# CORRELATION MATRIX
# ==========================================================

plt.figure(figsize=(10,8))

sns.heatmap(
    df.corr(),
    cmap='coolwarm',
    cbar=False
)

plt.title("Correlation Matrix")
plt.show()

# ==========================================================
# FEATURES & TARGET
# ==========================================================

X = df.drop(
    "default.payment.next.month",
    axis=1
)

y = df[
    "default.payment.next.month"
]

# ==========================================================
# TRAIN TEST SPLIT
# ==========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ==========================================================
# SMOTE
# ==========================================================

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print("\nAfter SMOTE")
print(pd.Series(y_train_smote).value_counts())

plt.figure(figsize=(6,4))
sns.countplot(x=y_train_smote)
plt.title("Class Distribution After SMOTE")
plt.show()

# ==========================================================
# FEATURE SCALING
# ==========================================================

scaler = StandardScaler()

X_train_smote = scaler.fit_transform(
    X_train_smote
)

X_test = scaler.transform(
    X_test
)

# ==========================================================
# MODELS
# ==========================================================

models = {

    "Logistic Regression":
    LogisticRegression(
        max_iter=2000
    ),

    "SVM":
    SVC(
        kernel='rbf',
        probability=True
    ),

    "Random Forest":
    RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        random_state=42
    ),

    "XGBoost":
    XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        random_state=42,
        eval_metric='logloss'
    )
}

# ==========================================================
# TRAINING & EVALUATION
# ==========================================================

results = []

best_accuracy = 0
best_model = None
best_model_name = ""

for name, model in models.items():

    print("\n")
    print("="*60)
    print(name)
    print("="*60)

    model.fit(
        X_train_smote,
        y_train_smote
    )

    y_pred = model.predict(
        X_test
    )

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    roc = roc_auc_score(
        y_test,
        y_pred
    )

    results.append(
        [name, accuracy, roc]
    )

    print(
        "\nAccuracy:",
        round(
            accuracy*100,
            2
        ),
        "%"
    )

    print(
        "\nROC-AUC:",
        round(
            roc,
            4
        )
    )

    print(
        classification_report(
            y_test,
            y_pred
        )
    )

    cm = confusion_matrix(
        y_test,
        y_pred
    )

    plt.figure(figsize=(5,4))

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues'
    )

    plt.title(
        f"Confusion Matrix - {name}"
    )

    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    plt.show()

    if accuracy > best_accuracy:

        best_accuracy = accuracy
        best_model = model
        best_model_name = name

# ==========================================================
# MODEL COMPARISON
# ==========================================================

results_df = pd.DataFrame(
    results,
    columns=[
        'Model',
        'Accuracy',
        'ROC_AUC'
    ]
)

results_df = results_df.sort_values(
    by='Accuracy',
    ascending=False
)

print("\nMODEL COMPARISON")
print(results_df)

plt.figure(figsize=(8,5))

sns.barplot(
    x='Model',
    y='Accuracy',
    data=results_df
)

plt.title("Model Comparison")
plt.xticks(rotation=15)
plt.show()

# ==========================================================
# FEATURE IMPORTANCE
# ==========================================================

if best_model_name in [
    "Random Forest",
    "XGBoost"
]:

    importance = best_model.feature_importances_

    importance_df = pd.DataFrame({

        'Feature': X.columns,
        'Importance': importance

    })

    importance_df = importance_df.sort_values(
        by='Importance',
        ascending=False
    )

    plt.figure(figsize=(10,6))

    sns.barplot(
        x='Importance',
        y='Feature',
        data=importance_df
    )

    plt.title(
        f"Feature Importance - {best_model_name}"
    )

    plt.show()

# ==========================================================
# SAVE BEST MODEL
# ==========================================================

joblib.dump(
    best_model,
    "best_credit_scoring_model.pkl"
)

print("\n")
print("="*60)
print("BEST MODEL :", best_model_name)
print(
    "BEST ACCURACY :",
    round(best_accuracy*100,2),
    "%"
)
print("="*60)

print("\nModel Saved Successfully")